In [4]:
import warnings;
warnings.simplefilter('ignore')
from dcr import *

In [5]:
%matplotlib inline
plt.rcParams['figure.dpi'] = 300
plt.rcParams['figure.figsize'] = (16,9)
plt.rcParams.update({'font.size': 16})

In [ ]:
PD = np.arange(0.0002, 0.05, 0.0002)
S = 5

rho_mortgages = 0.15*np.ones(size(PD))
rho_revolving = 0.04*np.ones(size(PD))
rho_retail = 0.03*(1-np.exp(-35*PD))/(1-np.exp(-35))+0.16*(1-(1-np.exp(-35*PD))/(1-np.exp(-35)))
rho_corporate = 0.12*(1-np.exp(-50*PD))/(1-np.exp(-50))+0.24*(1-(1-np.exp(-50*PD))/(1-np.exp(-50)))

sme_adjustment = 0.04*(1-max(S-5,0)/45)
rho_sme = rho_corporate - sme_adjustment

plt.plot(PD, rho_mortgages, label='Mortgages')
plt.plot(PD, rho_revolving, ls='dashed', label='Revolving')
plt.plot(PD, rho_retail, ls='dotted', label='Retail')
plt.plot(PD, rho_corporate, ls='dashdot', label='Corporate')
plt.plot(PD, rho_sme, marker='+', label='SME')
plt.xlabel("PD", fontsize=15)
plt.ylabel("rho", fontsize=15)
plt.tick_params(axis='both', labelsize=13)
plt.legend(loc='best')
plt.show()

In [ ]:
WCPD_mortgages = norm.cdf((norm.ppf(PD)+sqrt(rho_mortgages)*norm.ppf(0.999))/sqrt(1-rho_mortgages))
WCPD_revolving = norm.cdf((norm.ppf(PD)+sqrt(rho_revolving)*norm.ppf(0.999))/sqrt(1-rho_revolving))
WCPD_retail = norm.cdf((norm.ppf(PD)+sqrt(rho_retail)*norm.ppf(0.999))/sqrt(1-rho_retail))
WCPD_corporate = norm.cdf((norm.ppf(PD)+sqrt(rho_corporate)*norm.ppf(0.999))/sqrt(1-rho_corporate))
WCPD_sme = norm.cdf((norm.ppf(PD)+sqrt(rho_sme)*norm.ppf(0.999))/sqrt(1-rho_sme))

plt.plot(PD, WCPD_mortgages, label='Mortgages')
plt.plot(PD, WCPD_revolving, ls='dashed', label='Revolving')
plt.plot(PD, WCPD_retail, ls='dotted',  label='Retail')
plt.plot(PD, WCPD_corporate, ls='dashdot', label='Corporate')
plt.plot(PD, WCPD_sme, marker='+', label='SME')
plt.xlabel("PD", fontsize=15)
plt.ylabel("WCPD", fontsize=15)
plt.tick_params(axis='both', labelsize=13)
plt.legend(loc='best')
plt.show()

In [ ]:
default_rate = data.groupby("time")['default_time'].mean().reset_index(drop=False).rename(columns={"default_time": "default_rate"})

default_rate.loc[default_rate['default_rate'] <= 0, 'default_rate'] = 0.0001
default_rate.loc[default_rate['default_rate'] >= 1, 'default_rate'] = 0.9999

default_rate.loc[:, 'default_rate_p'] = scipy.stats.norm.ppf(default_rate.loc[:, 'default_rate'])

plt.subplots_adjust(hspace=0.4, wspace=0.4)

plt.subplot(221)
plt.plot('time', 'default_rate', data=default_rate)
plt.xlabel('time', fontsize=15)
plt.ylabel('default_rate', fontsize=15)
plt.tick_params(axis='both', labelsize=13)
plt.title('Default rate')

plt.subplot(222)
plt.plot('time', 'default_rate_p', data=default_rate)
plt.xlabel('time', fontsize=15)
plt.ylabel('default_rate_p', fontsize=15)
plt.tick_params(axis='both', labelsize=13)
plt.title('Probit of default rate')

plt.show()

In [ ]:
probitlinear = smf.ols(formula='default_rate_p ~ 1', data=default_rate).fit()

intercept = probitlinear.params.to_numpy().round(6)
mse = probitlinear.mse_resid
print('Intercept: ', np.asscalar(intercept))
print('MSE: ', mse.round(6))

In [ ]:
rho_hat = mse/(1+mse)
c_hat = np.asscalar(intercept) * (1-rho_hat)**0.5
PD_hat = norm.cdf(c_hat)

print('rho_hat:', rho_hat.round(3))
print('c_hat:', c_hat.round(3))
print('PD_hat:', PD_hat.round(3))

In [ ]:
dataCA = data.query('state_orig_time == "CA"')
dataNY = data.query('state_orig_time == "NY"')

default_rateCA = dataCA.groupby('time')['default_time'].mean().reset_index(drop=False).rename(columns={"default_time": "default_rate"})
default_rateNY = dataNY.groupby('time')['default_time'].mean().reset_index(drop=False).rename(columns={"default_time": "default_rate"})

default_rateCA.loc[default_rateCA['default_rate'] <= 0, 'default_rate'] = 0.0001
default_rateCA.loc[default_rateCA['default_rate'] >= 1, 'default_rate'] = 0.9999

default_rateNY.loc[default_rateNY['default_rate'] <= 0, 'default_rate'] = 0.0001
default_rateNY.loc[default_rateNY['default_rate'] >= 1, 'default_rate'] = 0.9999

default_rateCA.loc[:, 'default_rate_p'] = scipy.stats.norm.ppf(default_rateCA.loc[:, 'default_rate'])
default_rateNY.loc[:, 'default_rate_p'] = scipy.stats.norm.ppf(default_rateNY.loc[:, 'default_rate'])

plt.subplots_adjust(hspace=0.4, wspace=0.4)

plt.subplot(221)
plt.plot('time', 'default_rate', data=default_rateCA, label='CA')
plt.plot('time', 'default_rate', data=default_rateNY, color='red', ls='dashed', label='NY')
plt.xlabel('time', fontsize=15)
plt.ylabel('default_rate', fontsize=15)
plt.tick_params(axis='both', labelsize=13)
plt.legend(loc='best')
plt.title('Default rate')

plt.subplot(222)
plt.plot('time', 'default_rate_p', data=default_rateCA, label='CA')
plt.plot('time', 'default_rate_p', data=default_rateNY, color='red', ls='dashed', label='NY')
plt.xlabel('time', fontsize=15)
plt.ylabel('default_rate_p', fontsize=15)
plt.tick_params(axis='both', labelsize=13)
plt.legend(loc='best')
plt.title('Probit of default rate')
plt.show()


In [ ]:
probitlinear = smf.ols(formula='default_rate_p ~ 1', data=default_rateCA).fit()

intercept = probitlinear.params.to_numpy().round(6)
mse = probitlinear.mse_resid
rho_hat_CA = mse/(1+mse)
c_hat_CA = np.asscalar(intercept) * (1-rho_hat_CA)**0.5
PD_hat_CA = norm.cdf(c_hat_CA)

print('c_hat_CA:', c_hat_CA.round(3))
print('PD_hat_CA:', PD_hat_CA.round(3))
print('rho_hat_CA:', rho_hat_CA.round(3))


probitlinear = smf.ols(formula='default_rate_p ~ 1', data=default_rateNY).fit()

intercept = probitlinear.params.to_numpy().round(6)
mse = probitlinear.mse_resid
rho_hat_NY = mse/(1+mse)
c_hat_NY = np.asscalar(intercept) * (1-rho_hat_NY)**0.5
PD_hat_NY = norm.cdf(c_hat_NY)

print('rho_hat_NY:', rho_hat_NY.round(3))
print('c_hat_NY:', c_hat_NY.round(3))
print('PD_hat_NY:', PD_hat_NY.round(3))


In [ ]:
default_rate = data.groupby('time')[['default_time', 'gdp_time']].mean().reset_index(drop=False).rename(columns={"default_time": "default_rate"})

default_rate.loc[default_rate['default_rate'] <= 0, 'default_rate'] = 0.0001
default_rate.loc[default_rate['default_rate'] >= 1, 'default_rate'] = 0.9999

default_rate.loc[:, 'lag_defaultrate'] = default_rate['default_rate'].shift(1)

default_rate.loc[:, 'default_rate_p'] = scipy.stats.norm.ppf(default_rate.loc[:, 'default_rate'])
default_rate.loc[:, 'lag_defaultrate_p'] = scipy.stats.norm.ppf(default_rate.loc[:, 'lag_defaultrate'])

probitlinear = smf.ols(formula='default_rate_p ~ lag_defaultrate_p + gdp_time', data=default_rate).fit()

print(probitlinear.summary())

In [ ]:
fittedvalues = pd.DataFrame(probitlinear.fittedvalues, columns = ['default_rate_p_fit'])

default_rate = pd.merge(default_rate, fittedvalues, right_index=True, left_index=True)

validation(default_rate.default_rate_p_fit, default_rate.default_rate_p, default_rate.time, continuous=True)

In [ ]:
mse = probitlinear.mse_resid
rho_hat = mse/(1+mse)
print('rho_hat: ', rho_hat.round(3))

In [ ]:
PD1 = 0.01
PD2 = 0.05
rho1 = 0.1
rho2 = 0.2

CPD_range = np.arange(0.0005, 0.9995, 0.001)

g_CPD1 = ((1-rho1)**0.5)/((rho1)**0.5) * np.exp(0.5*(norm.ppf(CPD_range)**2) - 0.5/rho1 * (norm.ppf(PD1)-((1-rho1)**0.5) * norm.ppf(CPD_range))**2 )
g_CPD2 = ((1-rho1)**0.5)/((rho1)**0.5) * np.exp(0.5*(norm.ppf(CPD_range)**2) - 0.5/rho1 * (norm.ppf(PD2)-((1-rho1)**0.5) * norm.ppf(CPD_range))**2 )
g_CPD3 = ((1-rho2)**0.5)/((rho2)**0.5) * np.exp(0.5*(norm.ppf(CPD_range)**2) - 0.5/rho2 * (norm.ppf(PD1)-((1-rho2)**0.5) * norm.ppf(CPD_range))**2 )

G_CPD1 = norm.cdf((sqrt(1-rho1)*norm.ppf(CPD_range)-norm.ppf(PD1))/sqrt(rho1))
G_CPD2 = norm.cdf((sqrt(1-rho1)*norm.ppf(CPD_range)-norm.ppf(PD2))/sqrt(rho1))
G_CPD3 = norm.cdf((sqrt(1-rho2)*norm.ppf(CPD_range)-norm.ppf(PD1))/sqrt(rho2))

In [ ]:
def categorizeDensity(cum_density):
    cum_density_tmp1 = np.append(cum_density, 1)
    cum_density_tmp2 = np.roll(cum_density_tmp1, 1)
    cum_density_tmp2[0] = 0
    prob = cum_density_tmp1 - cum_density_tmp2
    return prob, cum_density_tmp1

prob_analytical1, cum_prob_analytical1 = categorizeDensity(G_CPD1)
prob_analytical2, cum_prob_analytical2 = categorizeDensity(G_CPD2)
prob_analytical3, cum_prob_analytical3 = categorizeDensity(G_CPD3)

CPD_range2 = np.append(CPD_range, 1)

In [ ]:
plt.subplots_adjust(hspace=0.4, wspace=0.4)

plt.subplot(221)
plt.plot(CPD_range2, prob_analytical1, label='PD=0.01')
plt.plot(CPD_range2, prob_analytical2, ls='dashed', label='PD=0.05')
plt.xlabel("Loss", fontsize=15)
xlim(0,0.1)
plt.ylabel("Probability", fontsize=15)
plt.tick_params(axis='both', labelsize=13)
plt.legend(loc='best', fontsize=15)
plt.title('Variation of PD - marginal')

plt.subplot(222)
plt.plot(CPD_range2, cum_prob_analytical1, label='PD=0.01')
plt.plot(CPD_range2, cum_prob_analytical2, ls='dashed', label='PD=0.05')
plt.xlabel("Loss", fontsize=15)
xlim(0,0.1)
plt.ylabel("Probability", fontsize=15)
plt.tick_params(axis='both', labelsize=13)
plt.legend(loc='best', fontsize=15)
plt.title('Variation of PD - cumulative')

plt.subplot(223)
plt.plot(CPD_range2, prob_analytical1, label='rho=0.1')
plt.plot(CPD_range2, prob_analytical3, ls='dashed', label='rho=0.2')
plt.xlabel("Loss", fontsize=15)
xlim(0,0.1)
plt.ylabel("Probability", fontsize=15)
plt.tick_params(axis='both', labelsize=13)
plt.legend(loc='best', fontsize=15)
plt.title('Variation of rho - marginal')

plt.subplot(224)
plt.plot(CPD_range2, cum_prob_analytical1, label='rho=0.1')
plt.plot(CPD_range2, cum_prob_analytical3, ls='dashed', label='rho=0.2')
plt.xlabel("Loss", fontsize=15)
xlim(0,0.1)
plt.ylabel("Probability", fontsize=15)
plt.tick_params(axis='both', labelsize=13)
plt.legend(loc='best', fontsize=15)
plt.title('Variation of rho - cumulative')
plt.show()

In [6]:
def numeric(PDin, rhoin, n, drlim):
    def integrand(F):
        return binom.pmf(d, n, norm.cdf((norm.ppf(PD)-(rho*0.5)*F)/((1-rho)**0.5))) * norm.pdf(F)

    default_rate_range = []
    prob = []
    for d in range(0,int(drlim*n)+1):
        PD = PDin
        rho = rhoin
        default_rate = d/n
        prob_temp = quad(integrand, -np.inf, np.inf)
        default_rate_range = np.append(default_rate_range,default_rate)
        prob = np.append(prob,prob_temp[0])
    cum_prob = np.cumsum(prob)
    return default_rate_range, prob, cum_prob

In [7]:
default_rate_range1, prob_numerical1, cum_prob_numerical1 = numeric(0.01, 0.1, 1000, 0.11)
default_rate_range2, prob_numerical2, cum_prob_numerical2 = numeric(0.05, 0.1, 1000, 0.11)
default_rate_range3, prob_numerical3, cum_prob_numerical3 = numeric(0.01, 0.2, 1000, 0.11)

: 

In [1]:
default_rate_range1, prob_numerical1, cum_prob_numerical1 = numeric(0.01, 0.1, 1000, 1)
default_rate_range2, prob_numerical2, cum_prob_numerical2 = numeric(0.05, 0.1, 1000, 1)
default_rate_range3, prob_numerical3, cum_prob_numerical3 = numeric(0.01, 0.2, 1000, 1)

NameError: name 'numeric' is not defined

In [2]:
plt.ylabel("Probability", fontsize=15)
plt.tick_params(axis='both', labelsize=13)
plt.legend(loc='best', fontsize=15)
plt.title('Variation of PD - marginal')

plt.subplot(222)
plt.plot(default_rate_range1, cum_prob_numerical1, label='PD=0.01')
plt.plot(default_rate_range2, cum_prob_numerical2, ls='dashed', label='PD=0.05')
plt.xlabel("Loss", fontsize=15)
xlim(0,0.1)
plt.ylabel("Probability", fontsize=15)
plt.tick_params(axis='both', labelsize=13)
plt.legend(loc='best', fontsize=15)
plt.title('Variation of PD - cumulative')

plt.subplot(223)
plt.plot(default_rate_range1, prob_numerical1, label='rho=0.1')
plt.plot(default_rate_range3, prob_numerical3, ls='dashed', label='rho=0.2')
plt.xlabel("Loss", fontsize=15)
xlim(0,0.1)
plt.ylabel("Probability", fontsize=15)
plt.tick_params(axis='both', labelsize=13)
plt.legend(loc='best', fontsize=15)
plt.title('Variation of rho - marginal')

plt.subplot(224)
plt.plot(default_rate_range1, cum_prob_numerical1, label='rho=0.1')
plt.plot(default_rate_range3, cum_prob_numerical3, ls='dashed', label='rho=0.2')
plt.xlabel("Loss", fontsize=15)
xlim(0,0.1)
plt.ylabel("Probability", fontsize=15)
plt.tick_params(axis='both', labelsize=13)
plt.legend(loc='best', fontsize=15)
plt.title('Variation of rho - cumulative')
plt.show()

NameError: name 'plt' is not defined

In [3]:
def simulation(PDin, rhoin, n, n_sim):
    F = np.random.normal(0, 1, n)
    output = []
    for i in F:
        defaultsum = 0
        eps = np.random.normal(0, 1, n)
        lnV = (rhoin**0.5) * i + ((1-rhoin)**0.5) * eps
        threshold = norm.ppf(PDin)
        default = lnV < threshold
        sumdefault = np.sum(default)
        defaultrate = sumdefault/n
        output = np.append(output, defaultrate)
    default_rate_range, counts = np.unique(output, return_counts=True)
    freq = counts/n_sim
    return default_rate_range, freq

In [4]:
default_rate_range4, freq1 = simulation(0.01, 0.1, 1000, 100000)
default_rate_range5, freq2 = simulation(0.05, 0.1, 1000, 100000)
default_rate_range6, freq3 = simulation(0.01, 0.2, 1000, 100000)

NameError: name 'np' is not defined

In [6]:
default_rate_range = pd.DataFrame({'default_rate_range1': default_rate_range1})
prob_simulation1 = pd.DataFrame({'freq1': freq1})
prob_simulation2 = pd.DataFrame({'freq2': freq2})
prob_simulation3 = pd.DataFrame({'freq3': freq3})

prob_simulation1 = default_rate_range.join(prob_simulation1).fillna(0).drop('default_rate_range1', axis='columns').to_numpy()
prob_simulation2 = default_rate_range.join(prob_simulation2).fillna(0).drop('default_rate_range1', axis='columns').to_numpy()
prob_simulation3 = default_rate_range.join(prob_simulation3).fillna(0).drop('default_rate_range1', axis='columns').to_numpy()

cum_prob_simulation1 = np.cumsum(prob_simulation1)
cum_prob_simulation2 = np.cumsum(prob_simulation2)
cum_prob_simulation3 = np.cumsum(prob_simulation3)

NameError: name 'pd' is not defined

In [7]:
plt.plot(default_rate_range1, cum_prob_simulation2, ls='dashed', label='PD=0.05')
plt.xlabel("Loss", fontsize=15)
plt.xlim(0,0.1)
plt.ylabel("Relative frequency", fontsize=15)
plt.tick_params(axis='both', labelsize=13)
plt.legend(loc='best', fontsize=15)
plt.title('Variation of PD - cumulative')

plt.subplot(223)
plt.plot(default_rate_range1, prob_simulation1, label='rho=0.1')
plt.plot(default_rate_range1, prob_simulation3, ls='dashed', label='rho=0.2')
plt.xlabel("Loss", fontsize=15)
plt.xlim(0,0.1)
plt.ylabel("Relative frequency", fontsize=15)
plt.tick_params(axis='both', labelsize=13)
plt.legend(loc='best', fontsize=15)
plt.title('Variation of rho - marginal')

plt.subplot(224)
plt.plot(default_rate_range1, cum_prob_simulation1, label='rho=0.1')
plt.plot(default_rate_range1, cum_prob_simulation3, ls='dashed', label='rho=0.2')
plt.xlabel("Loss", fontsize=15)
plt.xlim(0,0.1)
plt.ylabel("Relative frequency", fontsize=15)
plt.tick_params(axis='both', labelsize=13)
plt.legend(loc='best', fontsize=15)
plt.title('Variation of rho - cumulative')
plt.show()

NameError: name 'plt' is not defined

In [8]:
plt.subplots_adjust(hspace=0.4, wspace=0.4)

plt.subplot(221)
plt.plot(default_rate_range1, prob_analytical1, label='Analytical')
plt.plot(default_rate_range1, prob_numerical1, ls='dashed', label='Numerical')
plt.plot(default_rate_range1, prob_simulation1, ls='dotted', label='Simulation')
plt.xlabel("Loss", fontsize=15)
plt.xlim(0,0.1)
plt.ylabel("Probability, relative frequency", fontsize=15)
plt.tick_params(axis='both', labelsize=13)
plt.legend(loc='best', fontsize=15)
plt.title('Portfolio 1')

plt.subplot(222)
plt.plot(default_rate_range1, prob_analytical2, label='Analytical')
plt.plot(default_rate_range1, prob_numerical2, ls='dashed', label='Numerical')
plt.plot(default_rate_range1, prob_simulation2, ls='dotted', label='Simulation')
plt.xlabel("Loss", fontsize=15)
plt.xlim(0,0.1)
plt.ylabel("Probability, relative frequency", fontsize=15)
plt.tick_params(axis='both', labelsize=13)
plt.legend(loc='best', fontsize=15)
plt.title('Portfolio 2')

plt.subplot(223)
plt.plot(default_rate_range1, prob_analytical3, label='Analytical')
plt.plot(default_rate_range1, prob_numerical3, ls='dashed', label='Numerical')
plt.plot(default_rate_range1, prob_simulation3, ls='dotted', label='Simulation')
plt.xlabel("Loss", fontsize=15)
plt.xlim(0,0.1)
plt.ylabel("Probability, relative frequency", fontsize=15)
plt.tick_params(axis='both', labelsize=13)
plt.legend(loc='best', fontsize=15)
plt.title('Portfolio')
plt.show()

NameError: name 'plt' is not defined

In [9]:
expected_loss = [[0.01, 0.05, 0.01],
[np.dot(default_rate_range1, prob_analytical1), np.dot(default_rate_range1, prob_analytical2)
 , np.dot(default_rate_range1, prob_analytical3)],
[np.dot(default_rate_range1, prob_numerical1), np.dot(default_rate_range1, prob_numerical2),
 np.dot(default_rate_range1, prob_numerical3)],
[np.asscalar(np.dot(default_rate_range1, prob_simulation1)), np.asscalar(np.dot(
 default_rate_range1,prob_simulation2)), np.asscalar(np.dot(default_rate_range1,
 prob_simulation3))]]

np.round(expected_loss, 3)

NameError: name 'np' is not defined

In [10]:
alpha = 0.999

VaR_1 = norm.cdf((norm.ppf(0.01) + sqrt(0.1)*norm.ppf(alpha)) / sqrt(1-0.1))
VaR_2 = norm.cdf((norm.ppf(0.05) + sqrt(0.1)*norm.ppf(alpha)) / sqrt(1-0.1))
VaR_3 = norm.cdf((norm.ppf(0.01) + sqrt(0.2)*norm.ppf(alpha)) / sqrt(1-0.2))

print(np.round(VaR_1,3), np.round(VaR_2,3), np.round(VaR_3,3))

NameError: name 'norm' is not defined

In [11]:
def VaR(loss, cumprob, alpha):
    VaR = np.asscalar(loss[cumprob==min(cumprob, key=lambda x:abs(x-alpha))])
    return VaR

In [12]:
Value_at_Risk = [[VaR_1, VaR_2, VaR_3],
    [VaR(default_rate_range1, cum_prob_analytical1, alpha), VaR(default_rate_range1, cum_prob_analytical2, alpha), VaR(default_rate_range1, cum_prob_analytical3, alpha)],
    [VaR(default_rate_range1, cum_prob_numerical1, alpha), VaR(default_rate_range1, cum_prob_numerical2, alpha), VaR(default_rate_range1, cum_prob_numerical3, alpha)],
    [VaR(default_rate_range1, cum_prob_simulation1, alpha), VaR(default_rate_range1, cum_prob_simulation2, alpha), VaR(default_rate_range1, cum_prob_simulation3, alpha)]]

print(np.round(Value_at_Risk, 3))

NameError: name 'VaR_1' is not defined

In [13]:
def ES_ASRF(PD, rho, alpha):
    lower_boundary = np.array([-1000, -1000])
    upper_boundary = np.array([norm.ppf(PD), -norm.ppf(alpha)])
    mu = np.array([0, 0])
    cov = np.array([[1, sqrt(rho)], [sqrt(rho), 1]])
    BVNCDF_i = mvn.mvnun(lower_boundary, upper_boundary, mu, cov)
    ES_ASRF = BVNCDF_i / (1 - alpha)
    return ES_ASRF

ES_1 = ES_ASRF(0.01, 0.1, alpha)
ES_2 = ES_ASRF(0.05, 0.1, alpha)
ES_3 = ES_ASRF(0.01, 0.2, alpha)

print(np.round(ES_1, 3), np.round(ES_2, 3), np.round(ES_3, 3))

NameError: name 'np' is not defined

In [14]:
def ES(loss, prob, cumprob, alpha):
    VaR = np.asscalar(loss[cumprob == min(cumprob, key=lambda x:abs(x - alpha))])
    loss2 = loss[loss > VaR]
    prob2 = prob[loss > VaR]
    ES = np.asscalar(np.dot(loss2, prob2)) / (1 - alpha)
    return ES

In [15]:
expected_shortfall = [[ES_1, ES_2, ES_3],
[ES(default_rate_range1, prob_analytical1, cum_prob_analytical1, alpha),
 ES(default_rate_range1, prob_analytical2, cum_prob_analytical2, alpha),
 ES(default_rate_range1, prob_analytical3, cum_prob_analytical3, alpha)],
[ES(default_rate_range1, prob_numerical1, cum_prob_numerical1, alpha),
 ES(default_rate_range1, prob_numerical2, cum_prob_numerical2, alpha),
 ES(default_rate_range1, prob_numerical3, cum_prob_numerical3, alpha)],
[ES(default_rate_range1, prob_simulation1, cum_prob_simulation1, alpha),
 ES(default_rate_range1, prob_simulation2, cum_prob_simulation2, alpha),
 ES(default_rate_range1, prob_simulation3, cum_prob_simulation3, alpha)]]

print(np.round(expected_shortfall, 3))

NameError: name 'ES_1' is not defined